In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from src.morse_dataset import MorseDataset
from models.crnn import MorseCRNN1D
from src.train_model import train_model
from src.collate_fn import collate_fn, test_collate_fn
from src.encoding_decoding import decode_predictions, build_char_idx_dict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
df_train_full = pd.read_csv("data/train/labels.csv")
df_train_full["filename"] = df_train_full["filename"].map(lambda x: x.replace(".wav", ".pt"))
df_train, df_val = train_test_split(df_train_full, test_size=0.2, random_state=42)
df_train

,filename,text
21753,21753.pt,197
251,00251.pt,86
22941,22941.pt,94176
618,00618.pt,9535
17090,17090.pt,959
...,...,...
29802,29802.pt,07-8860
5390,05390.pt,039638
860,00860.pt,28-7
15795,15795.pt,5448374


In [3]:
alphabet = "0123456789- "
char_to_idx, idx_to_char = build_char_idx_dict(alphabet=alphabet)

train_files = ["data/train_specs_2/" + str(f) for f in df_train["filename"].values]
train_labels = df_train["text"].astype(str).values.tolist()

val_files = ["data/train_specs_2/" + str(f) for f in df_val["filename"].values]
val_labels = df_val["text"].astype(str).values.tolist()

train_dataset = MorseDataset(train_files, train_labels, char_to_idx)
val_dataset = MorseDataset(val_files, val_labels, char_to_idx)

In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

In [5]:
num_classes = len(char_to_idx)
model = MorseCRNN1D(num_classes=num_classes, model_type='lstm').to(device)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

model, history = train_model(
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    epochs=40,
    device=device,
    scheduler=scheduler,
    save_path="morse_crnn_lstm",
)

Epoch 1/40 | train_loss: 4.9722 | val_loss: 2.8737 | best_val_loss: 2.8737
Epoch 2/40 | train_loss: 2.7601 | val_loss: 2.6409 | best_val_loss: 2.6409
Epoch 3/40 | train_loss: 1.9270 | val_loss: 1.2519 | best_val_loss: 1.2519
Epoch 4/40 | train_loss: 0.8248 | val_loss: 0.6078 | best_val_loss: 0.6078
Epoch 5/40 | train_loss: 0.4815 | val_loss: 0.3529 | best_val_loss: 0.3529
Epoch 6/40 | train_loss: 0.3515 | val_loss: 0.3285 | best_val_loss: 0.3285
Epoch 7/40 | train_loss: 0.2859 | val_loss: 0.2427 | best_val_loss: 0.2427
Epoch 8/40 | train_loss: 0.2582 | val_loss: 0.2359 | best_val_loss: 0.2359
Epoch 9/40 | train_loss: 0.2398 | val_loss: 0.2319 | best_val_loss: 0.2319
Epoch 10/40 | train_loss: 0.2063 | val_loss: 0.2031 | best_val_loss: 0.2031
Epoch 11/40 | train_loss: 0.1913 | val_loss: 0.1913 | best_val_loss: 0.1913
Epoch 12/40 | train_loss: 0.1804 | val_loss: 0.2167 | best_val_loss: 0.1913
Epoch 13/40 | train_loss: 0.1814 | val_loss: 0.1818 | best_val_loss: 0.1818
Epoch 14/40 | train_l

In [8]:
submission_df = pd.read_csv("sample_submission.csv")
submission_df["filename"] = submission_df["filename"].map(lambda x: x.replace(".wav", ".pt"))

test_files = ["data/test_specs_2/" + str(f) for f in submission_df["filename"].values]
test_labels = [""] * len(test_files)

test_dataset = MorseDataset(test_files, test_labels, char_to_idx)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=test_collate_fn,
)

In [9]:
num_classes = len(char_to_idx)
model = MorseCRNN1D(num_classes=num_classes, model_type='lstm').to(device)

best_model_path = "morse_crnn_lstm_31_best.pt"
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

pred_texts = []

with torch.no_grad():
    for inputs, input_lengths in test_loader:
        inputs = inputs.to(device)

        outputs = model(inputs)

        if outputs.size(0) == inputs.size(0):
            outputs = outputs.permute(1, 0, 2)

        pred_ids = outputs.argmax(dim=2).cpu()

        output_time = outputs.size(0)
        input_time = inputs.size(-1)

        for i, length in enumerate(input_lengths):
            ids = pred_ids[:length, i].tolist()
            text = decode_predictions(ids, idx_to_char)
            pred_texts.append(text)

submission_df["text"] = pred_texts
submission_df["filename"] = submission_df["filename"].map(lambda x: x.replace('.pt', '.wav'))
submission_df.to_csv("submission_lstm.csv", index=False)